# Run RL / DL / ML & Monte Carlo in Colab

**Prerequisite:** Run `setup_colab.ipynb` first (GPU on, repo cloned, deps installed, `sys.path` set).

This notebook runs:
- **RL**: Volume PPO, Pattern PPO (stable-baselines3)
- **DL**: Pattern LSTM (PyTorch)
- **ML**: Regime classifier, pattern ML classifier (sklearn)
- **Monte Carlo Pro**: Strategy robustness (10k iterations)

In [ ]:
# Ensure project on path (if you ran setup in another runtime)
import sys, os
PROJECT_ROOT = "/content/claude"  # Must match setup_colab.ipynb
if os.path.exists(PROJECT_ROOT) and PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

## Synthetic OHLCV data (or replace with your CSV)

If you have US30/OHLCV data, upload a CSV and set `DATA_PATH` below. Otherwise we use synthetic data.

In [ ]:
import numpy as np
import pandas as pd

DATA_PATH = None  # e.g. "/content/drive/MyDrive/data/us30_ohlcv.csv"
np.random.seed(42)

if DATA_PATH and os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    df.columns = [c.strip().capitalize() for c in df.columns]
    print(f"Loaded {len(df)} rows from {DATA_PATH}")
else:
    n = 2000
    df = pd.DataFrame({
        "Open": 35000 + np.cumsum(np.random.randn(n) * 10),
        "Volume": np.random.randint(1000, 50000, n),
    })
    df["High"] = df["Open"] + np.abs(np.random.randn(n) * 20)
    df["Low"] = df["Open"] - np.abs(np.random.randn(n) * 20)
    df["Close"] = (df["High"] + df["Low"]) / 2 + np.random.randn(n) * 5
    df["returns"] = df["Close"].pct_change().fillna(0)
    print(f"Synthetic OHLCV: {len(df)} rows")

df.head()

## RL — Volume PPO (volume_rl_agent)

In [ ]:
from src.edges.volume_based.volume_rl_agent import train_rl_volume
from src.data.volume_loader import preprocess_volume_data

vol_df = df.copy()
if "Volume" not in vol_df.columns:
    vol_df["Volume"] = np.random.randint(1000, 50000, len(vol_df))
vol_processed, _, _ = preprocess_volume_data(vol_df)
if vol_processed.empty:
    vol_processed = vol_df[[c for c in ["Open", "High", "Low", "Close", "Volume", "returns"] if c in vol_df.columns]].fillna(0)
    vol_processed["spike_label"] = 0

model_vol = train_rl_volume(vol_processed, total_timesteps=5000)
print("Volume PPO:", "Trained" if model_vol else "Skipped (need stable_baselines3 + enough data)")

## RL — Pattern PPO (pattern_rl_agent)

In [ ]:
from src.edges.pattern_based.pattern_rl_agent import train_rl_pattern

pattern_df = df[[c for c in df.columns if c in ["Open", "High", "Low", "Close", "Volume", "returns"]]].fillna(0)
model_pattern = train_rl_pattern(pattern_df, total_timesteps=5000)
print("Pattern PPO:", "Trained" if model_pattern else "Skipped")

## DL — Pattern LSTM (pattern_detector_dl)

In [ ]:
from src.edges.pattern_based.pattern_detector_dl import train_pattern_lstm

seq_len = 20
cols = [c for c in ["Open", "High", "Low", "Close", "Volume"] if c in df.columns][:5]
X = df[cols].fillna(0).values
X_seq = np.array([X[i:i+seq_len] for i in range(len(X) - seq_len)])
y_labels = (np.sign(df["returns"].iloc[seq_len:].values) + 1).astype(int)
y_labels = np.clip(y_labels, 0, 2)

model_lstm = train_pattern_lstm(X_seq, y_labels, epochs=15)
print("Pattern LSTM:", "Trained" if model_lstm is not None else "Skipped (need torch)")

## ML — Regime classifier (Hurst + Random Forest)

In [ ]:
from src.ml.regime_classifier import classify_regime

if "Close" in df.columns and len(df) >= 150:
    df_regime = classify_regime(df)
    print("Regime column value counts:")
    print(df_regime["regime"].value_counts())
else:
    print("Need at least 150 rows with 'Close' for regime classification")

## Monte Carlo Pro — Strategy robustness (10k iterations)

In [ ]:
from src.tools.monte_carlo_pro import MonteCarloPro

returns = df["returns"] if "returns" in df.columns else df["Close"].pct_change().fillna(0)
returns = returns.replace([np.inf, -np.inf], 0).dropna()

mc = MonteCarloPro(iterations=10000, confidence_level=0.95)
sim = mc.simulate_paths(returns, initial_capital=100_000)
metrics = mc.get_decision_metrics(sim, initial_capital=100_000)
print(metrics)

## Optional: Save models to Drive

Uncomment and run if you mounted Google Drive and want to persist trained models.

In [ ]:
# from google.colab import drive
# import torch
# drive.mount("/content/drive")
# SAVE_DIR = "/content/drive/MyDrive/colab_models"
# os.makedirs(SAVE_DIR, exist_ok=True)
# if model_vol:
#     model_vol.save(os.path.join(SAVE_DIR, "ppo_volume.zip"))
# if model_pattern:
#     model_pattern.save(os.path.join(SAVE_DIR, "ppo_pattern.zip"))
# if model_lstm is not None:
#     torch.save(model_lstm.state_dict(), os.path.join(SAVE_DIR, "pattern_lstm.pt"))
# print("Saved to", SAVE_DIR)